# 02 - Extract GEE Features

Extract Google Earth Engine observations and create leakage-safe model feature and target files for sustainable agriculture land prediction.

## Leakage Rule

`t+1` values are written only to `gee_targets.csv` and `gee_targets_all.csv`. They are never stored in `gee_features.csv` or `gee_features_all.csv`.

In [36]:
from pathlib import Path
import importlib.util
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_INDEX_PATH = DATA_DIR / "processed" / "sample_index.csv"

# PowerShell environment variables are only inherited when the Jupyter kernel starts.
# If the env var is not visible here, set the project directly below.
GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "").strip() or "gen-lang-client-0671452157"

SAMPLE_LIMIT = None  # set to 1 for a quick smoke run
FORCE_REEXTRACT = False
VERBOSE_GEE_LOGS = True
LOG_FEATURE_GROUPS = True  # set False if the output is too noisy

print("Project root:", PROJECT_ROOT)
print("Sample index:", SAMPLE_INDEX_PATH)
print("GEE project:", GEE_PROJECT_ID)
print("Verbose GEE logs:", VERBOSE_GEE_LOGS)
print("Feature-group logs:", LOG_FEATURE_GROUPS)


Project root: c:\Users\Aki\Downloads\AML - Final Project\code
Sample index: c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\sample_index.csv
GEE project: gen-lang-client-0671452157
Verbose GEE logs: True
Feature-group logs: True


## Dependency Check

In [37]:
required_modules = {
    "ee": "earthengine-api",
    "geemap": "geemap",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
}
missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError("Missing packages. Run `pip install -r requirements.txt`: " + ", ".join(missing))


## Initialize Google Earth Engine

If you set `$env:GEE_PROJECT_ID` after launching Jupyter, restart the kernel or set `GEE_PROJECT_ID` directly in the config cell above.


In [38]:
import importlib
import src.features.gee_features as gee_feature_module

gee_feature_module = importlib.reload(gee_feature_module)
initialize_earth_engine = gee_feature_module.initialize_earth_engine

GEE_PROJECT_ID = (os.environ.get("GEE_PROJECT_ID", "").strip() or GEE_PROJECT_ID).strip()
if not GEE_PROJECT_ID:
    raise ValueError(
        "Set GEE_PROJECT_ID in the config cell, or restart Jupyter after running "
        "$env:GEE_PROJECT_ID='your-project-id' in PowerShell."
    )

ee = initialize_earth_engine(GEE_PROJECT_ID)
print("Earth Engine initialized with project:", GEE_PROJECT_ID)


Earth Engine initialized with project: gen-lang-client-0671452157


## Load Sample Index

In [39]:
from src.features.gee_features import load_sample_index

sample_index = load_sample_index(SAMPLE_INDEX_PATH)
print("Samples:", len(sample_index))
print(sample_index["label"].value_counts())
sample_index.head()


Samples: 10
label
moderate    4
high        3
low         3
Name: count, dtype: int64


,sample_id,label,latitude,longitude,processed_dir,frame_count,start_date,end_date,frame_metadata_csv,processing_metadata_json,gee_observations_csv,gee_features_csv,gee_targets_csv,gee_feature_metadata_json
0,3.3908_102.01211_high,high,3.39080,102.01211,c:\Users\Aki\Downloads\AML - Final Project\cod...,18,2016-06-08,2025-10-04,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...
1,3.40614_102.05138_high,high,3.40614,102.05138,c:\Users\Aki\Downloads\AML - Final Project\cod...,18,2016-06-08,2025-10-04,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...
2,3.4521_102.0134_high,high,3.45210,102.01340,c:\Users\Aki\Downloads\AML - Final Project\cod...,18,2016-06-08,2025-10-04,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...
3,3.44071_101.66564_low,low,3.44071,101.66564,c:\Users\Aki\Downloads\AML - Final Project\cod...,47,2016-01-30,2026-06-06,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...
4,3.46234_101.64491_low,low,3.46234,101.64491,c:\Users\Aki\Downloads\AML - Final Project\cod...,51,2016-01-30,2026-06-06,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...,c:\Users\Aki\Downloads\AML - Final Project\cod...


## Progress Output

The extraction cell prints sample, frame-date, and feature-group progress with elapsed seconds. If output is too noisy, set `LOG_FEATURE_GROUPS = False` in the config cell.


## Extract GEE Observations, Features, and Targets

In [40]:
import importlib
import src.features.gee_features as gee_feature_module

gee_feature_module = importlib.reload(gee_feature_module)
FeatureExtractionConfig = gee_feature_module.FeatureExtractionConfig
extract_all_samples = gee_feature_module.extract_all_samples

config = FeatureExtractionConfig(
    gee_project_id=GEE_PROJECT_ID,
    data_dir=str(DATA_DIR),
    sample_index_csv=str(SAMPLE_INDEX_PATH),
    buffer_radius_m=1000,
    context_buffer_radius_m=5000,
    s1_tolerance_days=15,
    force=FORCE_REEXTRACT,
    verbose=VERBOSE_GEE_LOGS,
    log_feature_groups=LOG_FEATURE_GROUPS,
)

gee_features_all, gee_targets_all = extract_all_samples(
    config=config,
    sample_limit=SAMPLE_LIMIT,
    ee_module=ee,
)

print("Feature rows:", len(gee_features_all))
print("Target rows:", len(gee_targets_all))
gee_features_all.head()


Starting GEE extraction: 10 sample(s), project=gen-lang-client-0671452157
Outputs will be written under: c:\Users\Aki\Downloads\AML - Final Project\code\data\processed


GEE samples:   0%|          | 0/10 [00:00<?, ?it/s]

[1/10] 3.3908_102.01211_high: start
    3.3908_102.01211_high: using cached observations from c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\high\3.3908_102.01211\gee_observations.csv
[1/10] 3.3908_102.01211_high: building temporal features/targets
[1/10] 3.3908_102.01211_high: done (18 frame rows, 0.1s)
[2/10] 3.40614_102.05138_high: start
    3.40614_102.05138_high: using cached observations from c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\high\3.40614_102.05138\gee_observations.csv
[2/10] 3.40614_102.05138_high: building temporal features/targets
[2/10] 3.40614_102.05138_high: done (18 frame rows, 0.0s)
[3/10] 3.4521_102.0134_high: start
    3.4521_102.0134_high: using cached observations from c:\Users\Aki\Downloads\AML - Final Project\code\data\processed\high\3.4521_102.0134\gee_observations.csv
[3/10] 3.4521_102.0134_high: building temporal features/targets
[3/10] 3.4521_102.0134_high: done (18 frame rows, 0.0s)
[4/10] 3.44071_101.66564_low: star

,sample_id,label,latitude,longitude,frame_index,acquisition_date,ndvi_lag_1,ndvi_lag_2,ndvi_rolling_mean_3,ndvi_trend,evi_lag_1,evi_lag_2,evi_rolling_mean_3,evi_trend,built_growth_rate,built_growth_trend,rainfall_rolling_mean_3,heavy_rain_rolling_sum_3,sar_moisture_trend,flood_risk_proxy_score
0,3.3908_102.01211_high,high,3.3908,102.01211,0,2016-06-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,240.494630,1.0,NaN,0.017500
1,3.3908_102.01211_high,high,3.3908,102.01211,1,2017-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.010219,-0.010219,242.362122,3.0,NaN,0.277070
2,3.3908_102.01211_high,high,3.3908,102.01211,2,2018-08-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008733,-0.000743,203.773270,4.0,-0.930565,0.113640
3,3.3908_102.01211_high,high,3.3908,102.01211,3,2019-09-01,NaN,NaN,0.610006,NaN,NaN,NaN,0.504218,NaN,-0.005065,-0.002184,170.963211,3.0,-0.792465,0.207459
4,3.3908_102.01211_high,high,3.3908,102.01211,4,2019-09-06,0.610006,NaN,0.601388,NaN,0.504218,NaN,0.503068,NaN,0.000469,0.001379,143.698152,1.0,-0.124427,0.191814


## Verify Leakage Separation

In [41]:
from src.features.gee_features import assert_no_target_leakage

assert_no_target_leakage(gee_features_all)
assert not any(column.startswith("target_") for column in gee_features_all.columns)
assert any(column.startswith("target_") for column in gee_targets_all.columns)
print("Leakage check passed: target columns are physically separated from model features.")


Leakage check passed: target columns are physically separated from model features.


## Output Files

Per sample: `gee_observations.csv`, `gee_features.csv`, `gee_targets.csv`, and `gee_feature_metadata.json`.

Combined: `data/processed/gee_features_all.csv` and `data/processed/gee_targets_all.csv`.